# Data / Model Init

In [1]:
# N_SAMPLES = 10000
N_SAMPLES = None
random_state = 11
fold_id = 0

FILTERS = {
    "duration_unit": ["h"],
    "effect": ["MOR", "POP", "GRO", "BEH", "REP", "ITX", "PHY", "DVP", "MPH"],
}

SPLIT_SALTS = False
REMOVE_LONE = False
REMOVE_METALS = False

MAX_CONC_VALUE = 10000
DURATION_FILL_VALUE = 1e-6
MAX_DURATION_HOURS = 9000.0

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    possible = Path.cwd() / "vollmers/gnn-thesis/gnn-thesis"
    if (possible / "src").exists():
        PROJECT_ROOT = possible

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch_geometric.loader import DataLoader
from torch_geometric.utils.smiles import from_smiles

from src.data.io import load_data
from src.data.cleaning import process_data, print_mol_types
from src.data.graph_building import build_graphs
from src.data.metadata import sequential_encoder, build_config
from src.data.cleaning import fragment_count, is_salt, has_metal, is_single_node
from src.data.splitting import butina_split, show_split_info
from src.data.sampling import LoadData, show_loader_info, display_sampling_effect
from src.training.loops import train
from src.visualization.training_plots import  plot_training, plot_training_metrics, plot_group_training


pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)

print("Imports imported")


Imports imported


In [2]:
DATA_PATH = PROJECT_ROOT / "Data" / "toxicity_all.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

print(f"Data path: {DATA_PATH}")

df_all = load_data(DATA_PATH)

df_processed = process_data(
    df_all,
    n_samples=N_SAMPLES,
    random_state=random_state,
    filters=FILTERS,
    require_duration=False,
    require_taxid=True,
    split_salts=SPLIT_SALTS,
    remove_lone=REMOVE_LONE,
    remove_metals=REMOVE_METALS,
    max_conc_value=MAX_CONC_VALUE,
    duration_fill_value=DURATION_FILL_VALUE,
    max_duration_hours=MAX_DURATION_HOURS,
    log_transform_duration=True,
    keep_duration_raw=True,
)

df_processed


Data path: /cephyr/users/vollmers/Alvis/vollmers/gnn-thesis/gnn-thesis/Data/toxicity_all.csv
Filters
conc > 0 and <= 10000
True: 0.990
duration_unit: ['h']
True: 0.980
effect: ['MOR', 'POP', 'GRO', 'BEH', 'REP', 'ITX', 'PHY', 'DVP', 'MPH']
True: 0.981
require_taxid: True
True: 0.995
duration <= 9000.0 h or missing
True: 0.986

Loaded and masked training data
Rows in full data: 561,100
Rows after mask: 525,852
Rows before preprocessing: 525,852
Rows after preprocessing:  525,852
Rows removed: 0


,SK_unique_id,species_common_name,species_latin_name,CAS,chemical_name,conc_unit,conc,duration,duration_unit,effect,endpoint,SMILES,organism_lifestage,administration_route,species_sci_name,taxid,superkingdom,kingdom,phylum,subphylum,class,order,family,genus,species,species_group,log10c,duration_raw
0,RTECS7,mouse,Mus musculus,108073-64-9,"5H-Pyrrolo(2,1-c)(1,4)benzodiazepin-5-one, 1,2,3,10,11,11a-hexahydro-2-hydro...",mg/kg,36.200,-6.000000,h,MOR,EC50,COC1Nc2ccccc2C(=O)N2CC(O)CC12,adult,fill,mus musculus,10090,2759.0,33208.0,7711.0,89593.0,40674.0,9989.0,10066.0,10088.0,10090.0,rodents,1.558709,NaN
1,RTECS41,mouse,Mus musculus,78111-17-8,"Acanthifolicin, 9,10-deepithio-9,10-didehydro-",mg/kg,0.192,-6.000000,h,MOR,EC50,C=C1C(O)C2OC3(CCC(C=CC(C)C4CC(C)=CC5(OC(CC(C)(O)C(=O)O)CCC5O)O4)O3)CCC2OC1C(...,adult,fill,mus musculus,10090,2759.0,33208.0,7711.0,89593.0,40674.0,9989.0,10066.0,10088.0,10090.0,rodents,-0.716699,NaN
2,RTECS137,mouse,Mus musculus,4657-93-6,5-Acenaphthenamine,mg/kg,56.000,-6.000000,h,MOR,EC50,Nc1ccc2c3c(cccc13)CC2,adult,fill,mus musculus,10090,2759.0,33208.0,7711.0,89593.0,40674.0,9989.0,10066.0,10088.0,10090.0,rodents,1.748188,NaN
3,RTECS140,mouse,Mus musculus,102585-20-6,"5-Acenaphthenamine, 6,7,8,8a-tetrahydro-N-(2-oxazolin-2-yl)-",mg/kg,100.000,-6.000000,h,MOR,EC50,c1cc(NC2=NCCO2)c2c3c1CCC3CCC2,adult,fill,mus musculus,10090,2759.0,33208.0,7711.0,89593.0,40674.0,9989.0,10066.0,10088.0,10090.0,rodents,2.000000,NaN
4,RTECS141,rat,Rattus norvegicus,83-32-9,Acenaphthene,mg/kg,600.000,-6.000000,h,MOR,EC50,c1cc2c3c(cccc3c1)CC2,adult,fill,rattus norvegicus,10116,2759.0,33208.0,7711.0,89593.0,40674.0,9989.0,10066.0,10114.0,10116.0,rodents,2.778151,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525847,AQTER1441956,Eelgrass,vallisneria natans,7758-98-7,Sulfuric acid copper(2+) salt (1:1),mg/L,0.041,2.283301,h,GRO,LOEC,O=S(=O)([O-])[O-].[Cu+2],adult,renewal,vallisneria natans,62345,2759.0,33090.0,35493.0,131221.0,3398.0,16360.0,26319.0,26336.0,62345.0,plants,-1.387216,192.0
525848,AQTER1441958,Eelgrass,vallisneria natans,7758-98-7,Sulfuric acid copper(2+) salt (1:1),mg/L,0.640,1.982271,h,GRO,LOEC,O=S(=O)([O-])[O-].[Cu+2],adult,renewal,vallisneria natans,62345,2759.0,33090.0,35493.0,131221.0,3398.0,16360.0,26319.0,26336.0,62345.0,plants,-0.193820,96.0
525849,AQTER1441960,Eelgrass,vallisneria natans,7758-98-7,Sulfuric acid copper(2+) salt (1:1),mg/L,0.160,1.982271,h,GRO,NOEC,O=S(=O)([O-])[O-].[Cu+2],adult,renewal,vallisneria natans,62345,2759.0,33090.0,35493.0,131221.0,3398.0,16360.0,26319.0,26336.0,62345.0,plants,-0.795880,96.0
525850,AQTER1441966,Eelgrass,vallisneria natans,7758-98-7,Sulfuric acid copper(2+) salt (1:1),mg/L,0.640,1.982271,h,GRO,NOEC,O=S(=O)([O-])[O-].[Cu+2],adult,renewal,vallisneria natans,62345,2759.0,33090.0,35493.0,131221.0,3398.0,16360.0,26319.0,26336.0,62345.0,plants,-0.193820,96.0


In [3]:
from src.data.simple_featurizer import simple_featurizer
from src.data.features_graph import GraphFeaturizer

ATOM_FEATURES = (
    "atomic_num",
    "degree",
    "formal_charge",
    "num_hs",
    "hybridization",
    "is_aromatic",
    "is_in_ring",
    "atomic_mass",
    "period",
    "group",
    "covalent_radius",
    "vdw_radius",
    "is_metal",
)

BOND_FEATURES = (
    "bond_order",
    "is_conjugated",
    "is_in_ring",
    "stereo",
)

graph_featurizer = GraphFeaturizer(ATOM_FEATURES, BOND_FEATURES)

df_processed["features"] = df_processed["SMILES"].apply(graph_featurizer.featurize)

graph_cache = graph_featurizer.get_graph_cache()

from src.data.features_mol import add_molecule_metadata

MOLECULE_CATEGORICAL_COLS = [
    "is_salt",
    "has_metal",
    "is_single_node",
]

MOLECULE_NUMERICAL_COLS = [
    "fragment_count",
    # "mol_weight",
    "log10_mol_weight",
    # "logp",
    # "tpsa",
    # "log10_tpsa_plus1",
    # "h_bond_donor_count",
    # "h_bond_acceptor_count",
    # "heavy_atom_count",
    # "log10_heavy_atom_count_plus1",
    # "hetero_atom_count",
    # "halogen_count",
    # "metal_count",
    # "transition_metal_count",
    # "ring_count",
    # "aromatic_ring_count",
    # "rotatable_bond_count",
    "formal_charge",
]

# Add molecule-level metadata for categorical and numerical encoders
df_processed = add_molecule_metadata(df_processed, categorical_cols=MOLECULE_CATEGORICAL_COLS, numerical_cols=MOLECULE_NUMERICAL_COLS)

USE_PRETRAINED_TAXID = True
PRETRAINED_TAXID_PATH = PROJECT_ROOT / "Data" / "moredata" / "pretrained_tax_emb.pkl.zip"
print(f"Pretrained taxid path: {PRETRAINED_TAXID_PATH}")

config_tax = {}

# Categorical encoding
exp_categorical_cols = [
    "species_group",
    "conc_unit",
    "endpoint", 
    "effect"
]

CATEGORICAL_COLS = exp_categorical_cols + MOLECULE_CATEGORICAL_COLS

df_categorical = df_processed[CATEGORICAL_COLS].copy()
df_categorical, categorical_encoder = sequential_encoder(df_categorical, CATEGORICAL_COLS)
# df_categorical now contains only the sequential data for selected columns

config_categorical = build_config(df_categorical, CATEGORICAL_COLS)

species_group_decoder = {encoded: original for original, encoded in categorical_encoder["species_group"].items()}

NUMERICAL_COLS = ["duration"] + MOLECULE_NUMERICAL_COLS

graphs = build_graphs(
    df_processed,  
    df_categorical,
    CATEGORICAL_COLS,
    NUMERICAL_COLS,
    )

from src.data.splitting import _build_dataset

full_dataset = _build_dataset(graphs, range(len(graphs)))

Pretrained taxid path: /cephyr/users/vollmers/Alvis/vollmers/gnn-thesis/gnn-thesis/Data/moredata/pretrained_tax_emb.pkl.zip


In [5]:
import re

EXPERIMENT_DIR = PROJECT_ROOT / "outputs" / "experiments"
fold_pattern = re.compile(r"fold_(\d+)_val_predictions\.csv\.gz$")


def load_model_folds(model_name):
    model_dir = EXPERIMENT_DIR / model_name
    frames = []

    for path in sorted(model_dir.glob("fold_*_val_predictions.csv.gz")):
        fold = int(fold_pattern.search(path.name).group(1))

        df = pd.read_csv(path, compression="gzip")
        df["fold"] = fold
        df["model"] = model_name

        frames.append(df)

    if not frames:
        raise FileNotFoundError(f"No fold prediction files found in {model_dir}")

    return pd.concat(frames, ignore_index=True)

afp_df = load_model_folds("afp-11M-10fold")

In [6]:
loader = LoadData(
    dataset=full_dataset, 
    batch_size=256, 
    sampler_type="sequential",
    shuffle=False, 
    attribute="species_group",
    target_dataset=full_dataset
)

In [7]:
from src.models.pna import compute_pna_degree_histogram

from src.models.grapefruit import Grapefruit
from src.models.attentive_fp import AttentiveFP
from src.models.toxicity_model import ToxicityModel
from src.models.meta_encoder import MetaEncoder


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

########## MODEL HYPERPARAMETERS ##########

PRETRAINED_TAX_DIM = 768 # 768 is the length of the vectors in pretrained_tax_emb.pkl.zip
PRETRAINED_TAXID_OUTPUT_DIM = 512
CATEGORICAL_DIM = 128
NUMERIC_DIM = 128
META_DROPOUT = 0.3

GNN_HIDDEN_DIM = 512
GNN_OUT_DIM = 512
TOWERS = 4

NUM_LAYERS = 3
NUM_TIMESTEPS = 2
DROPOUT = 0.3

FINAL_HIDDEN_DIM = 1024

ATOM_FEATURE_DIM = graphs[0].x.shape[1]
EDGE_FEATURE_DIM = graphs[0].edge_attr.shape[1]
VIRTUAL_EDGE_FEATURE_DIM = graphs[0].virtual_edge_attr.shape[1] if hasattr(graphs[0], "virtual_edge_attr") else 0

def build_model():
    meta_encoder = MetaEncoder(
        pretrained_taxid_path=PRETRAINED_TAXID_PATH if USE_PRETRAINED_TAXID else None,
        pretrained_tax_dim=PRETRAINED_TAX_DIM,
        pretrained_taxid_output_dim=PRETRAINED_TAXID_OUTPUT_DIM,
        config_categorical=config_categorical,
        categorical_output_dim=CATEGORICAL_DIM,
        numerical_columns=NUMERICAL_COLS,
        numeric_output_dim=NUMERIC_DIM,
        dropout=META_DROPOUT
    ).to(device)

    # meta_encoder = None

    # pna_deg = compute_pna_degree_histogram(full_dataset)

    # model_gnn = Grapefruit(
    #     in_channels=ATOM_FEATURE_DIM,
    #     edge_dim=EDGE_FEATURE_DIM,
    #     virtual_edge_dim=VIRTUAL_EDGE_FEATURE_DIM,
    #     hidden_dim=GNN_HIDDEN_DIM,
    #     towers=TOWERS,
    #     deg=pna_deg,
    #     out_dim=GNN_OUT_DIM,
    #     num_layers=NUM_LAYERS,
    #     num_timesteps=NUM_TIMESTEPS,
    #     dropout=DROPOUT,
    # ).to(device)

    model_gnn = AttentiveFP(
        in_channels=ATOM_FEATURE_DIM,
        edge_dim=EDGE_FEATURE_DIM,
        hidden_channels=GNN_HIDDEN_DIM,
        out_channels=GNN_OUT_DIM,
        num_layers=NUM_LAYERS,
        num_timesteps=NUM_TIMESTEPS,
        dropout=0.3,
    ).to(device)

    # model_gnn = None

    model = ToxicityModel(
        model_gnn,
        meta_encoder,
        hidden_dim=FINAL_HIDDEN_DIM,
    ).to(device)

    n_params_meta = sum(p.numel() for p in meta_encoder.parameters() if p.requires_grad) if meta_encoder else 0
    n_params_gnn = sum(p.numel() for p in model_gnn.parameters() if p.requires_grad) if model_gnn else 0
    n_params_total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    gnn_name = type(model_gnn).__name__ if model_gnn is not None else "metadata_only"

    return model, meta_encoder, model_gnn, n_params_meta, n_params_gnn, n_params_total, gnn_name

model, meta_encoder, model_gnn, *_ = build_model()

# checkpoint_path = PROJECT_ROOT / "outputs" / "models" / "Grapefruit-9M-150-fold0.pt"

# checkpoint_path = PROJECT_ROOT / "outputs" / "models" / "AFP-11M-150" / "AttentiveFP-11M-150-fold0.pt"


# checkpoint = torch.load(checkpoint_path, map_location=device)

# model.load_state_dict(checkpoint["model_state_dict"])
# model = model.to(device)

# model.eval()

# print(f"Loaded model from {checkpoint_path}")
# print(f"Best epoch: {checkpoint.get('best_epoch')}")
# print(f"Best score: {checkpoint.get('best_monitor_value')}")

# Interpretability

In [8]:
from torch_geometric.loader import DataLoader


AVG_MODEL_DIR = PROJECT_ROOT / "outputs" / "models" / "afp-11M-10fold"
AVG_CHECKPOINT_TEMPLATE = "AttentiveFP-11M-fold{fold}.pt"
AVG_GNN_BATCH_SIZE = 512
AVG_VAL_BATCH_SIZE = 512


def first_unique_smiles_indices(df, smiles_col="SMILES"):
    """Return graph-list positions for the first row of each unique molecule."""
    if smiles_col not in df.columns:
        raise KeyError(f"{smiles_col!r} is missing from the dataframe.")

    work = df[[smiles_col]].copy()
    work["_graph_index"] = np.arange(len(work), dtype=int)
    unique_rows = work.dropna(subset=[smiles_col]).drop_duplicates(smiles_col)
    return unique_rows["_graph_index"].astype(int).tolist()


UNIQUE_MOLECULE_INDICES = first_unique_smiles_indices(df_processed)
print(f"Unique molecules for average GNN embedding: {len(UNIQUE_MOLECULE_INDICES):,}")


def graph_loader_from_indices(indices, batch_size=512, shuffle=False):
    dataset = [graphs[int(idx)] for idx in indices]
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def load_fold_model(fold):
    model, *_ = build_model()
    checkpoint_path = AVG_MODEL_DIR / AVG_CHECKPOINT_TEMPLATE.format(fold=int(fold))

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()

    print(f"Loaded model from {checkpoint_path}")
    return model, checkpoint, checkpoint_path


@torch.no_grad()
def compute_average_gnn_embedding(model, molecule_indices=None, batch_size=AVG_GNN_BATCH_SIZE):
    if model.gnn is None:
        raise ValueError("The model has no GNN encoder to average.")

    molecule_indices = UNIQUE_MOLECULE_INDICES if molecule_indices is None else molecule_indices
    loader = graph_loader_from_indices(molecule_indices, batch_size=batch_size, shuffle=False)

    model.eval()
    embedding_sum = None
    n_embeddings = 0

    for batch in loader:
        batch = batch.to(device)
        gnn_embedding = model.gnn(batch).detach()

        batch_sum = gnn_embedding.sum(dim=0)
        embedding_sum = batch_sum if embedding_sum is None else embedding_sum + batch_sum
        n_embeddings += gnn_embedding.shape[0]

    if n_embeddings == 0:
        raise ValueError("No molecule embeddings were produced.")

    return (embedding_sum / n_embeddings).detach().cpu()


def fold_validation_frame(fold):
    fold_df = afp_df.loc[afp_df["fold"].astype(int).eq(int(fold))].copy()
    if fold_df.empty:
        raise ValueError(f"No validation rows found in afp_df for fold {fold}.")

    fold_df["row_id"] = fold_df["row_id"].astype(int)
    return fold_df.reset_index(drop=True)


@torch.no_grad()
def predict_with_average_gnn(model, row_ids, avg_gnn_embedding, batch_size=AVG_VAL_BATCH_SIZE):
    loader = graph_loader_from_indices(row_ids, batch_size=batch_size, shuffle=False)
    avg_gnn_embedding = avg_gnn_embedding.to(device).view(1, -1)

    frames = []
    model.eval()

    for batch in loader:
        batch = batch.to(device)
        normal_gnn_embedding = model.gnn(batch)
        avg_batch_embedding = avg_gnn_embedding.expand(normal_gnn_embedding.shape[0], -1)

        if model.meta_encoder is None:
            normal_combined = normal_gnn_embedding
            avg_combined = avg_batch_embedding
        else:
            meta_embedding = model.meta_encoder(batch)
            normal_combined = torch.cat([normal_gnn_embedding, meta_embedding], dim=1)
            avg_combined = torch.cat([avg_batch_embedding, meta_embedding], dim=1)

        pred = model.predictor(normal_combined).view(-1)
        pred_avgmol = model.predictor(avg_combined).view(-1)

        frames.append(
            pd.DataFrame(
                {
                    "row_id": batch.row_id.detach().cpu().view(-1).numpy().astype(int),
                    "actual_from_graph": batch.y.detach().cpu().view(-1).numpy(),
                    "pred_log10c": pred.detach().cpu().numpy(),
                    "pred_avgmol_log10c": pred_avgmol.detach().cpu().numpy(),
                }
            )
        )

    return pd.concat(frames, ignore_index=True)


def run_average_molecule_ablation(
    folds=None,
    molecule_indices=None,
    gnn_batch_size=AVG_GNN_BATCH_SIZE,
    val_batch_size=AVG_VAL_BATCH_SIZE,
):
    folds = sorted(afp_df["fold"].dropna().astype(int).unique()) if folds is None else list(folds)
    molecule_indices = UNIQUE_MOLECULE_INDICES if molecule_indices is None else molecule_indices

    fold_results = []

    for fold in folds:
        model, checkpoint, checkpoint_path = load_fold_model(fold)
        avg_gnn_embedding = compute_average_gnn_embedding(
            model,
            molecule_indices=molecule_indices,
            batch_size=gnn_batch_size,
        )

        fold_df = fold_validation_frame(fold)
        pred_df = predict_with_average_gnn(
            model,
            fold_df["row_id"].tolist(),
            avg_gnn_embedding,
            batch_size=val_batch_size,
        )

        fold_meta = fold_df.rename(
            columns={
                "pred_log10c": "saved_pred_log10c",
                "abs_error_log10c": "saved_abs_error_log10c",
            }
        )

        merged = fold_meta.merge(pred_df, on="row_id", how="left", validate="one_to_one")
        merged["fold"] = int(fold)
        merged["checkpoint_path"] = str(checkpoint_path)
        merged["n_average_molecules"] = len(molecule_indices)

        target_col = "actual_log10c" if "actual_log10c" in merged.columns else "log10c"
        merged["abs_error_log10c"] = (merged["pred_log10c"] - merged[target_col]).abs()
        merged["avgmol_abs_error_log10c"] = (merged["pred_avgmol_log10c"] - merged[target_col]).abs()
        merged["mol_effect_log10c"] = merged["pred_log10c"] - merged["pred_avgmol_log10c"]
        merged["abs_mol_effect_log10c"] = merged["mol_effect_log10c"].abs()
        merged["avgmol_error_minus_normal_error"] = merged["avgmol_abs_error_log10c"] - merged["abs_error_log10c"]

        if "saved_pred_log10c" in merged.columns:
            merged["pred_minus_saved_log10c"] = merged["pred_log10c"] - merged["saved_pred_log10c"]

        fold_results.append(merged)

        print(
            f"Fold {fold}: normal median AE={merged['abs_error_log10c'].median():.4f}, "
            f"avg-molecule median AE={merged['avgmol_abs_error_log10c'].median():.4f}"
        )

        del model, checkpoint, avg_gnn_embedding
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.concat(fold_results, ignore_index=True)


def avgmol_long_results(results_df):
    base_cols = [
        col
        for col in ["fold", "row_id", "endpoint", "species_group", "SMILES", "actual_log10c"]
        if col in results_df.columns
    ]

    normal = results_df[base_cols].copy()
    normal["prediction_type"] = "normal"
    normal["pred_log10c"] = results_df["pred_log10c"].to_numpy()
    normal["abs_error_log10c"] = results_df["abs_error_log10c"].to_numpy()

    avgmol = results_df[base_cols].copy()
    avgmol["prediction_type"] = "average_molecule"
    avgmol["pred_log10c"] = results_df["pred_avgmol_log10c"].to_numpy()
    avgmol["abs_error_log10c"] = results_df["avgmol_abs_error_log10c"].to_numpy()

    long_df = pd.concat([normal, avgmol], ignore_index=True)
    long_df["endpoint"] = long_df["endpoint"].astype("string").fillna("Missing").astype(str)
    long_df["species_group"] = long_df["species_group"].astype("string").fillna("Missing").astype(str)
    return long_df


def summarize_avgmol_endpoint_species(results_df, min_count_per_fold=1):
    long_df = avgmol_long_results(results_df)

    fold_summary = (
        long_df.groupby(["fold", "endpoint", "species_group", "prediction_type"], dropna=False)
        .agg(
            median_ae=("abs_error_log10c", "median"),
            mean_ae=("abs_error_log10c", "mean"),
            n=("abs_error_log10c", "size"),
        )
        .reset_index()
    )

    if min_count_per_fold is not None:
        fold_summary = fold_summary[fold_summary["n"] >= min_count_per_fold].copy()

    def sem(values):
        values = pd.Series(values).dropna()
        if len(values) <= 1:
            return 0.0
        return values.std(ddof=1) / np.sqrt(len(values))

    summary = (
        fold_summary.groupby(["endpoint", "species_group", "prediction_type"], dropna=False)
        .agg(
            median_ae_mean=("median_ae", "mean"),
            median_ae_std=("median_ae", "std"),
            median_ae_sem=("median_ae", sem),
            median_ae_min=("median_ae", "min"),
            median_ae_max=("median_ae", "max"),
            n_folds=("median_ae", "size"),
            total_n=("n", "sum"),
        )
        .reset_index()
    )
    summary["median_ae_std"] = summary["median_ae_std"].fillna(0.0)

    delta = summary.pivot_table(
        index=["endpoint", "species_group"],
        columns="prediction_type",
        values="median_ae_mean",
    )
    if {"normal", "average_molecule"}.issubset(delta.columns):
        delta = (delta["average_molecule"] - delta["normal"]).rename("avgmol_minus_normal_median_ae")
        summary = summary.merge(delta.reset_index(), on=["endpoint", "species_group"], how="left")

    return fold_summary, summary


def plot_avgmol_endpoint_species_median_ae(
    results_df,
    endpoints=None,
    top_species=8,
    min_total_n=20,
    min_count_per_fold=1,
    errorbar="std",
    figsize=None,
):
    errorbar = str(errorbar).lower()
    error_cols = {"std": "median_ae_std", "sem": "median_ae_sem", "none": None}
    if errorbar not in error_cols:
        raise ValueError("errorbar must be one of: 'std', 'sem', 'none'.")

    fold_summary, summary = summarize_avgmol_endpoint_species(
        results_df,
        min_count_per_fold=min_count_per_fold,
    )

    if endpoints is None:
        endpoints = (
            results_df["endpoint"]
            .astype("string")
            .fillna("Missing")
            .value_counts()
            .index.astype(str)
            .tolist()
        )
    else:
        endpoints = [str(endpoint) for endpoint in endpoints]

    if figsize is None:
        figsize = (12, max(3.5, 0.55 * top_species * max(len(endpoints), 1)))

    fig, axes = plt.subplots(len(endpoints), 1, figsize=figsize, sharex=True)
    axes = np.atleast_1d(axes)
    error_col = error_cols[errorbar]
    type_labels = {"normal": "Normal", "average_molecule": "Average molecule"}
    colors = {"normal": "#2f6db3", "average_molecule": "#d9822b"}

    for ax, endpoint in zip(axes, endpoints):
        endpoint_summary = summary[summary["endpoint"].eq(endpoint)].copy()

        species_counts = (
            endpoint_summary.groupby("species_group")["total_n"]
            .max()
            .sort_values(ascending=False)
        )
        if min_total_n is not None:
            species_counts = species_counts[species_counts >= min_total_n]
        species_order = species_counts.head(top_species).index.tolist()

        if not species_order:
            ax.text(0.5, 0.5, f"No species groups for {endpoint}", ha="center", va="center")
            ax.set_axis_off()
            continue

        y = np.arange(len(species_order))
        bar_height = 0.36

        for offset, prediction_type in [(-bar_height / 2, "normal"), (bar_height / 2, "average_molecule")]:
            plot_df = (
                endpoint_summary[endpoint_summary["prediction_type"].eq(prediction_type)]
                .set_index("species_group")
                .reindex(species_order)
            )
            values = plot_df["median_ae_mean"].astype(float).fillna(0.0).to_numpy()
            xerr = None
            if error_col is not None:
                xerr = plot_df[error_col].astype(float).fillna(0.0).to_numpy()

            ax.barh(
                y + offset,
                values,
                height=bar_height,
                xerr=xerr,
                capsize=3 if xerr is not None else 0,
                label=type_labels[prediction_type],
                color=colors[prediction_type],
                alpha=0.9,
            )

        labels = [f"{species} (n={int(species_counts.loc[species])})" for species in species_order]
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.invert_yaxis()
        ax.set_title(f"{endpoint}: median AE by species group")
        ax.grid(axis="x", alpha=0.25)

    axes[-1].set_xlabel("Fold-mean median absolute error in log10 concentration")
    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

    return fig, axes, summary


# Computes one avg mol embedding for each fold based on all molecules
# Each folds model predicts on the corresponding fold validatoin set 

avgmol_results_df = run_average_molecule_ablation()
avgmol_fold_summary, avgmol_endpoint_species_summary = summarize_avgmol_endpoint_species(avgmol_results_df)
fig, axes, avgmol_plot_summary = plot_avgmol_endpoint_species_median_ae(
    avgmol_results_df,
    top_species=8,
    min_total_n=20,
    errorbar="std",
)

avgmol_endpoint_species_summary.sort_values(
    ["endpoint", "avgmol_minus_normal_median_ae", "total_n"],
    ascending=[True, False, False],
).head(30)

Unique molecules for average GNN embedding: 80,535
Loaded model from /cephyr/users/vollmers/Alvis/vollmers/gnn-thesis/gnn-thesis/outputs/models/afp-11M-10fold/AttentiveFP-11M-fold0.pt
Fold 0: normal median AE=0.4603, avg-molecule median AE=0.8774
Loaded model from /cephyr/users/vollmers/Alvis/vollmers/gnn-thesis/gnn-thesis/outputs/models/afp-11M-10fold/AttentiveFP-11M-fold1.pt
Fold 1: normal median AE=0.5035, avg-molecule median AE=0.9661
Loaded model from /cephyr/users/vollmers/Alvis/vollmers/gnn-thesis/gnn-thesis/outputs/models/afp-11M-10fold/AttentiveFP-11M-fold2.pt
Fold 2: normal median AE=0.4753, avg-molecule median AE=0.8875
Loaded model from /cephyr/users/vollmers/Alvis/vollmers/gnn-thesis/gnn-thesis/outputs/models/afp-11M-10fold/AttentiveFP-11M-fold3.pt
Fold 3: normal median AE=0.4866, avg-molecule median AE=0.9003
Loaded model from /cephyr/users/vollmers/Alvis/vollmers/gnn-thesis/gnn-thesis/outputs/models/afp-11M-10fold/AttentiveFP-11M-fold4.pt
Fold 4: normal median AE=0.5039,

,endpoint,species_group,prediction_type,median_ae_mean,median_ae_std,median_ae_sem,median_ae_min,median_ae_max,n_folds,total_n,avgmol_minus_normal_median_ae
34,EC10,unmapped,average_molecule,1.338694,0.437277,0.138279,0.679652,2.011018,10,392,0.900673
35,EC10,unmapped,normal,0.438021,0.094599,0.029915,0.341324,0.676897,10,392,0.900673
6,EC10,cnidarians and bryozoans,average_molecule,1.148765,0.772619,0.244324,0.156308,2.564235,10,74,0.649134
7,EC10,cnidarians and bryozoans,normal,0.499631,0.342362,0.108264,0.031510,1.070180,10,74,0.649134
12,EC10,echinoderms,average_molecule,1.415883,0.728757,0.230453,0.413027,2.608420,10,83,0.636680
13,EC10,echinoderms,normal,0.779204,0.425524,0.134563,0.315481,1.599608,10,83,0.636680
14,EC10,fish,average_molecule,1.199588,0.423771,0.134008,0.672222,2.116405,10,2054,0.600863
15,EC10,fish,normal,0.598725,0.159403,0.050408,0.383782,0.882444,10,2054,0.600863
18,EC10,insects,average_molecule,1.214314,0.298279,0.094324,0.898536,1.645227,10,1088,0.577422
19,EC10,insects,normal,0.636891,0.213647,0.067561,0.377895,1.148103,10,1088,0.577422


In [9]:
from torch_geometric.loader import DataLoader


AVGTAXID_MODEL_DIR = PROJECT_ROOT / "outputs" / "models" / "afp-11M-10fold"
AVGTAXID_CHECKPOINT_TEMPLATE = "AttentiveFP-11M-fold{fold}.pt"
AVGTAXID_BATCH_SIZE = 512


def species_group_key(value):
    if pd.isna(value):
        return "Missing"
    return str(value)


def first_unique_taxid_indices(df, taxid_col="taxid"):
    """Return graph-list positions for the first row of each unique raw taxid."""
    if taxid_col not in df.columns:
        raise KeyError(f"{taxid_col!r} is missing from the dataframe.")

    work = df[[taxid_col]].copy()
    work["_graph_index"] = np.arange(len(work), dtype=int)
    work["_taxid_key"] = pd.to_numeric(work[taxid_col], errors="coerce")
    work = work.dropna(subset=["_taxid_key"]).copy()
    work["_taxid_key"] = work["_taxid_key"].astype(int)
    work = work[work["_taxid_key"] > 0]

    unique_rows = work.drop_duplicates("_taxid_key")
    return unique_rows["_graph_index"].astype(int).tolist()


def first_unique_taxid_indices_by_species_group(df, taxid_col="taxid", group_col="species_group"):
    """Return graph-list positions for unique raw taxids within each species group."""
    missing_cols = {taxid_col, group_col}.difference(df.columns)
    if missing_cols:
        raise KeyError(f"Missing columns: {sorted(missing_cols)}")

    work = df[[taxid_col, group_col]].copy()
    work["_graph_index"] = np.arange(len(work), dtype=int)
    work["_species_group_key"] = work[group_col].map(species_group_key)
    work["_taxid_key"] = pd.to_numeric(work[taxid_col], errors="coerce")
    work = work.dropna(subset=["_taxid_key"]).copy()
    work["_taxid_key"] = work["_taxid_key"].astype(int)
    work = work[work["_taxid_key"] > 0]

    return {
        species_group: group_df.drop_duplicates("_taxid_key")["_graph_index"].astype(int).tolist()
        for species_group, group_df in work.groupby("_species_group_key", dropna=False)
    }


UNIQUE_TAXID_INDICES = first_unique_taxid_indices(df_processed)
UNIQUE_TAXID_INDICES_BY_SPECIES_GROUP = first_unique_taxid_indices_by_species_group(df_processed)
print(f"Unique taxids for average pretrained taxid embedding: {len(UNIQUE_TAXID_INDICES):,}")
print(
    "Unique taxids by species group: "
    + ", ".join(
        f"{species_group}={len(indices):,}"
        for species_group, indices in sorted(UNIQUE_TAXID_INDICES_BY_SPECIES_GROUP.items())
    )
)


def taxid_graph_loader_from_indices(indices, batch_size=AVGTAXID_BATCH_SIZE, shuffle=False):
    dataset = [graphs[int(idx)] for idx in indices]
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def load_fold_model_for_taxid(fold):
    model, *_ = build_model()
    checkpoint_path = AVGTAXID_MODEL_DIR / AVGTAXID_CHECKPOINT_TEMPLATE.format(fold=int(fold))

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()

    print(f"Loaded model from {checkpoint_path}")
    return model, checkpoint, checkpoint_path


def fold_validation_frame_for_taxid(fold):
    fold_df = afp_df.loc[afp_df["fold"].astype(int).eq(int(fold))].copy()
    if fold_df.empty:
        raise ValueError(f"No validation rows found in afp_df for fold {fold}.")

    fold_df["row_id"] = fold_df["row_id"].astype(int)
    return fold_df.reset_index(drop=True)


@torch.no_grad()
def compute_average_pretrained_taxid_embedding(model, taxid_indices=None, batch_size=AVGTAXID_BATCH_SIZE):
    meta_encoder = model.meta_encoder
    if meta_encoder is None or meta_encoder.pretrained_taxid_encoder is None:
        raise ValueError("The model has no pretrained taxid encoder to average.")

    taxid_indices = UNIQUE_TAXID_INDICES if taxid_indices is None else taxid_indices
    loader = taxid_graph_loader_from_indices(taxid_indices, batch_size=batch_size, shuffle=False)

    model.eval()
    embedding_sum = None
    n_embeddings = 0

    for batch in loader:
        batch = batch.to(device)
        taxid_embedding = meta_encoder.pretrained_taxid_encoder(batch).detach()

        batch_sum = taxid_embedding.sum(dim=0)
        embedding_sum = batch_sum if embedding_sum is None else embedding_sum + batch_sum
        n_embeddings += taxid_embedding.shape[0]

    if n_embeddings == 0:
        raise ValueError("No taxid embeddings were produced.")

    return (embedding_sum / n_embeddings).detach().cpu()


def compute_average_pretrained_taxid_embeddings_by_species_group(
    model,
    taxid_indices_by_species_group=None,
    batch_size=AVGTAXID_BATCH_SIZE,
):
    taxid_indices_by_species_group = (
        UNIQUE_TAXID_INDICES_BY_SPECIES_GROUP
        if taxid_indices_by_species_group is None
        else taxid_indices_by_species_group
    )

    return {
        species_group_key(species_group): compute_average_pretrained_taxid_embedding(
            model,
            taxid_indices=indices,
            batch_size=batch_size,
        )
        for species_group, indices in taxid_indices_by_species_group.items()
        if len(indices) > 0
    }


def meta_embedding_with_average_taxid(meta_encoder, batch, avg_taxid_embedding):
    if meta_encoder is None or meta_encoder.pretrained_taxid_encoder is None:
        raise ValueError("A pretrained taxid encoder is required for this ablation.")

    batch_size = batch.y.view(-1).numel()
    avg_taxid_embedding = avg_taxid_embedding.to(device)
    if avg_taxid_embedding.dim() == 1:
        avg_taxid_embedding = avg_taxid_embedding.view(1, -1).expand(batch_size, -1)
    elif avg_taxid_embedding.dim() == 2:
        if avg_taxid_embedding.shape[0] == 1:
            avg_taxid_embedding = avg_taxid_embedding.expand(batch_size, -1)
        elif avg_taxid_embedding.shape[0] != batch_size:
            raise ValueError(
                "A batched average taxid embedding must have one row per graph in the batch."
            )
    else:
        raise ValueError("Average taxid embedding must be a 1D or 2D tensor.")

    encoded_parts = []
    if meta_encoder.tax_encoder is not None:
        encoded_parts.append(meta_encoder.tax_encoder(batch))

    encoded_parts.append(avg_taxid_embedding)

    if meta_encoder.categorical_encoder is not None:
        encoded_parts.append(meta_encoder.categorical_encoder(batch))

    if meta_encoder.numeric_encoder is not None:
        encoded_parts.append(meta_encoder.numeric_encoder(batch))

    return torch.cat(encoded_parts, dim=-1)


def combine_toxicity_inputs(gnn_embedding=None, meta_embedding=None):
    embeddings = [embedding for embedding in [gnn_embedding, meta_embedding] if embedding is not None]
    if not embeddings:
        raise ValueError("At least one embedding is required for prediction.")
    if len(embeddings) == 1:
        return embeddings[0]
    return torch.cat(embeddings, dim=1)


def species_group_average_taxid_batch_embedding(
    row_ids,
    avg_taxid_embeddings_by_species_group,
    default_embedding,
):
    if avg_taxid_embeddings_by_species_group is None:
        return None

    row_ids = np.asarray(row_ids, dtype=int)
    species_groups = df_processed.iloc[row_ids]["species_group"].map(species_group_key).tolist()
    embeddings = [
        avg_taxid_embeddings_by_species_group.get(species_group, default_embedding)
        for species_group in species_groups
    ]
    return torch.stack([embedding.detach().cpu() for embedding in embeddings], dim=0)


@torch.no_grad()
def predict_with_average_taxid(
    model,
    row_ids,
    avg_taxid_embedding,
    avg_taxid_embeddings_by_species_group=None,
    batch_size=AVGTAXID_BATCH_SIZE,
):
    loader = taxid_graph_loader_from_indices(row_ids, batch_size=batch_size, shuffle=False)

    frames = []
    model.eval()

    for batch in loader:
        batch = batch.to(device)
        batch_row_ids = batch.row_id.detach().cpu().view(-1).numpy().astype(int)

        gnn_embedding = model.gnn(batch) if model.gnn is not None else None
        normal_meta_embedding = model.meta_encoder(batch) if model.meta_encoder is not None else None
        avg_taxid_meta_embedding = meta_embedding_with_average_taxid(
            model.meta_encoder,
            batch,
            avg_taxid_embedding,
        )
        species_group_avg_taxid_embedding = species_group_average_taxid_batch_embedding(
            batch_row_ids,
            avg_taxid_embeddings_by_species_group,
            avg_taxid_embedding,
        )
        species_group_avg_taxid_meta_embedding = None
        if species_group_avg_taxid_embedding is not None:
            species_group_avg_taxid_meta_embedding = meta_embedding_with_average_taxid(
                model.meta_encoder,
                batch,
                species_group_avg_taxid_embedding,
            )

        normal_combined = combine_toxicity_inputs(gnn_embedding, normal_meta_embedding)
        avg_taxid_combined = combine_toxicity_inputs(gnn_embedding, avg_taxid_meta_embedding)

        pred = model.predictor(normal_combined).view(-1)
        pred_avgtaxid = model.predictor(avg_taxid_combined).view(-1)

        frame_data = {
            "row_id": batch_row_ids,
            "actual_from_graph": batch.y.detach().cpu().view(-1).numpy(),
            "pred_log10c": pred.detach().cpu().numpy(),
            "pred_avgtaxid_log10c": pred_avgtaxid.detach().cpu().numpy(),
        }

        if species_group_avg_taxid_meta_embedding is not None:
            species_group_avg_taxid_combined = combine_toxicity_inputs(
                gnn_embedding,
                species_group_avg_taxid_meta_embedding,
            )
            pred_species_group_avgtaxid = model.predictor(species_group_avg_taxid_combined).view(-1)
            frame_data["pred_species_group_avgtaxid_log10c"] = (
                pred_species_group_avgtaxid.detach().cpu().numpy()
            )

        frames.append(pd.DataFrame(frame_data))

    return pd.concat(frames, ignore_index=True)


def run_average_taxid_ablation(
    folds=None,
    taxid_indices=None,
    taxid_indices_by_species_group=None,
    include_species_group_average=True,
    batch_size=AVGTAXID_BATCH_SIZE,
):
    folds = sorted(afp_df["fold"].dropna().astype(int).unique()) if folds is None else list(folds)
    taxid_indices = UNIQUE_TAXID_INDICES if taxid_indices is None else taxid_indices
    if taxid_indices_by_species_group is None:
        taxid_indices_by_species_group = UNIQUE_TAXID_INDICES_BY_SPECIES_GROUP
    else:
        taxid_indices_by_species_group = {
            species_group_key(species_group): list(indices)
            for species_group, indices in taxid_indices_by_species_group.items()
        }
    n_taxids_by_species_group = {
        species_group: len(indices)
        for species_group, indices in taxid_indices_by_species_group.items()
    }

    fold_results = []

    for fold in folds:
        model, checkpoint, checkpoint_path = load_fold_model_for_taxid(fold)
        avg_taxid_embedding = compute_average_pretrained_taxid_embedding(
            model,
            taxid_indices=taxid_indices,
            batch_size=batch_size,
        )
        avg_taxid_embeddings_by_species_group = None
        if include_species_group_average:
            avg_taxid_embeddings_by_species_group = compute_average_pretrained_taxid_embeddings_by_species_group(
                model,
                taxid_indices_by_species_group=taxid_indices_by_species_group,
                batch_size=batch_size,
            )

        fold_df = fold_validation_frame_for_taxid(fold)
        pred_df = predict_with_average_taxid(
            model,
            fold_df["row_id"].tolist(),
            avg_taxid_embedding,
            avg_taxid_embeddings_by_species_group=avg_taxid_embeddings_by_species_group,
            batch_size=batch_size,
        )

        fold_meta = fold_df.rename(
            columns={
                "pred_log10c": "saved_pred_log10c",
                "abs_error_log10c": "saved_abs_error_log10c",
            }
        )

        merged = fold_meta.merge(pred_df, on="row_id", how="left", validate="one_to_one")
        merged["fold"] = int(fold)
        merged["checkpoint_path"] = str(checkpoint_path)
        merged["n_average_taxids"] = len(taxid_indices)

        target_col = "actual_log10c" if "actual_log10c" in merged.columns else "log10c"
        merged["abs_error_log10c"] = (merged["pred_log10c"] - merged[target_col]).abs()
        merged["avgtaxid_abs_error_log10c"] = (merged["pred_avgtaxid_log10c"] - merged[target_col]).abs()
        merged["taxid_effect_log10c"] = merged["pred_log10c"] - merged["pred_avgtaxid_log10c"]
        merged["abs_taxid_effect_log10c"] = merged["taxid_effect_log10c"].abs()
        merged["avgtaxid_error_minus_normal_error"] = merged["avgtaxid_abs_error_log10c"] - merged["abs_error_log10c"]

        if "pred_species_group_avgtaxid_log10c" in merged.columns:
            merged["species_group_avgtaxid_abs_error_log10c"] = (
                merged["pred_species_group_avgtaxid_log10c"] - merged[target_col]
            ).abs()
            merged["species_group_taxid_effect_log10c"] = (
                merged["pred_log10c"] - merged["pred_species_group_avgtaxid_log10c"]
            )
            merged["abs_species_group_taxid_effect_log10c"] = merged[
                "species_group_taxid_effect_log10c"
            ].abs()
            merged["species_group_avgtaxid_error_minus_normal_error"] = (
                merged["species_group_avgtaxid_abs_error_log10c"] - merged["abs_error_log10c"]
            )
            species_group_keys = merged["species_group"].map(species_group_key)
            merged["n_average_taxids_species_group"] = (
                species_group_keys.map(n_taxids_by_species_group)
                .fillna(len(taxid_indices))
                .astype(int)
            )

        if "saved_pred_log10c" in merged.columns:
            merged["pred_minus_saved_log10c"] = merged["pred_log10c"] - merged["saved_pred_log10c"]

        fold_results.append(merged)

        message = (
            f"Fold {fold}: normal median AE={merged['abs_error_log10c'].median():.4f}, "
            f"avg-taxid median AE={merged['avgtaxid_abs_error_log10c'].median():.4f}"
        )
        if "species_group_avgtaxid_abs_error_log10c" in merged.columns:
            message += (
                ", species-group avg-taxid median AE="
                f"{merged['species_group_avgtaxid_abs_error_log10c'].median():.4f}"
            )
        print(message)

        del model, checkpoint, avg_taxid_embedding, avg_taxid_embeddings_by_species_group
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.concat(fold_results, ignore_index=True)


def avgtaxid_long_results(results_df):
    base_cols = [
        col
        for col in ["fold", "row_id", "endpoint", "species_group", "taxid", "actual_log10c"]
        if col in results_df.columns
    ]

    normal = results_df[base_cols].copy()
    normal["prediction_type"] = "normal"
    normal["pred_log10c"] = results_df["pred_log10c"].to_numpy()
    normal["abs_error_log10c"] = results_df["abs_error_log10c"].to_numpy()

    avgtaxid = results_df[base_cols].copy()
    avgtaxid["prediction_type"] = "average_taxid"
    avgtaxid["pred_log10c"] = results_df["pred_avgtaxid_log10c"].to_numpy()
    avgtaxid["abs_error_log10c"] = results_df["avgtaxid_abs_error_log10c"].to_numpy()

    frames = [normal, avgtaxid]
    if {
        "pred_species_group_avgtaxid_log10c",
        "species_group_avgtaxid_abs_error_log10c",
    }.issubset(results_df.columns):
        species_group_avgtaxid = results_df[base_cols].copy()
        species_group_avgtaxid["prediction_type"] = "species_group_average_taxid"
        species_group_avgtaxid["pred_log10c"] = results_df[
            "pred_species_group_avgtaxid_log10c"
        ].to_numpy()
        species_group_avgtaxid["abs_error_log10c"] = results_df[
            "species_group_avgtaxid_abs_error_log10c"
        ].to_numpy()
        frames.append(species_group_avgtaxid)

    long_df = pd.concat(frames, ignore_index=True)
    long_df["endpoint"] = long_df["endpoint"].astype("string").fillna("Missing").astype(str)
    long_df["species_group"] = long_df["species_group"].astype("string").fillna("Missing").astype(str)
    return long_df


def summarize_avgtaxid_endpoint_species(results_df, min_count_per_fold=1):
    long_df = avgtaxid_long_results(results_df)

    fold_summary = (
        long_df.groupby(["fold", "endpoint", "species_group", "prediction_type"], dropna=False)
        .agg(
            median_ae=("abs_error_log10c", "median"),
            mean_ae=("abs_error_log10c", "mean"),
            n=("abs_error_log10c", "size"),
        )
        .reset_index()
    )

    if min_count_per_fold is not None:
        fold_summary = fold_summary[fold_summary["n"] >= min_count_per_fold].copy()

    def sem(values):
        values = pd.Series(values).dropna()
        if len(values) <= 1:
            return 0.0
        return values.std(ddof=1) / np.sqrt(len(values))

    summary = (
        fold_summary.groupby(["endpoint", "species_group", "prediction_type"], dropna=False)
        .agg(
            median_ae_mean=("median_ae", "mean"),
            median_ae_std=("median_ae", "std"),
            median_ae_sem=("median_ae", sem),
            median_ae_min=("median_ae", "min"),
            median_ae_max=("median_ae", "max"),
            n_folds=("median_ae", "size"),
            total_n=("n", "sum"),
        )
        .reset_index()
    )
    summary["median_ae_std"] = summary["median_ae_std"].fillna(0.0)

    delta = summary.pivot_table(
        index=["endpoint", "species_group"],
        columns="prediction_type",
        values="median_ae_mean",
    )
    delta_specs = [
        ("average_taxid", "avgtaxid_minus_normal_median_ae"),
        ("species_group_average_taxid", "species_group_avgtaxid_minus_normal_median_ae"),
    ]
    for prediction_type, delta_col in delta_specs:
        if {"normal", prediction_type}.issubset(delta.columns):
            prediction_delta = (delta[prediction_type] - delta["normal"]).rename(delta_col)
            summary = summary.merge(
                prediction_delta.reset_index(),
                on=["endpoint", "species_group"],
                how="left",
            )

    return fold_summary, summary


def plot_avgtaxid_endpoint_species_median_ae(
    results_df,
    endpoints=None,
    top_species=8,
    min_total_n=20,
    min_count_per_fold=1,
    errorbar="std",
    figsize=None,
):
    errorbar = str(errorbar).lower()
    error_cols = {"std": "median_ae_std", "sem": "median_ae_sem", "none": None}
    if errorbar not in error_cols:
        raise ValueError("errorbar must be one of: 'std', 'sem', 'none'.")

    fold_summary, summary = summarize_avgtaxid_endpoint_species(
        results_df,
        min_count_per_fold=min_count_per_fold,
    )

    if endpoints is None:
        endpoints = (
            results_df["endpoint"]
            .astype("string")
            .fillna("Missing")
            .value_counts()
            .index.astype(str)
            .tolist()
        )
    else:
        endpoints = [str(endpoint) for endpoint in endpoints]

    if figsize is None:
        figsize = (12, max(3.5, 0.55 * top_species * max(len(endpoints), 1)))

    fig, axes = plt.subplots(len(endpoints), 1, figsize=figsize, sharex=True)
    axes = np.atleast_1d(axes)
    error_col = error_cols[errorbar]
    type_labels = {
        "normal": "Normal",
        "average_taxid": "Average taxid (all)",
        "species_group_average_taxid": "Average taxid (group)",
    }
    colors = {
        "normal": "#2f6db3",
        "average_taxid": "#7b51b8",
        "species_group_average_taxid": "#2f8f5b",
    }
    available_prediction_types = set(summary["prediction_type"].astype(str))
    prediction_order = [
        prediction_type
        for prediction_type in type_labels
        if prediction_type in available_prediction_types
    ]
    n_unique_taxids_by_species_group = {
        species_group_key(species_group): len(indices)
        for species_group, indices in UNIQUE_TAXID_INDICES_BY_SPECIES_GROUP.items()
    }

    for ax, endpoint in zip(axes, endpoints):
        endpoint_summary = summary[summary["endpoint"].eq(endpoint)].copy()

        species_counts = (
            endpoint_summary.groupby("species_group")["total_n"]
            .max()
            .sort_values(ascending=False)
        )
        if min_total_n is not None:
            species_counts = species_counts[species_counts >= min_total_n]
        species_order = species_counts.head(top_species).index.tolist()

        if not species_order:
            ax.text(0.5, 0.5, f"No species groups for {endpoint}", ha="center", va="center")
            ax.set_axis_off()
            continue

        y = np.arange(len(species_order))
        bar_height = min(0.36, 0.8 / max(len(prediction_order), 1))
        offsets = (
            np.arange(len(prediction_order)) - (len(prediction_order) - 1) / 2
        ) * bar_height

        for offset, prediction_type in zip(offsets, prediction_order):
            plot_df = (
                endpoint_summary[endpoint_summary["prediction_type"].eq(prediction_type)]
                .set_index("species_group")
                .reindex(species_order)
            )
            values = plot_df["median_ae_mean"].astype(float).fillna(0.0).to_numpy()
            xerr = None
            if error_col is not None:
                xerr = plot_df[error_col].astype(float).fillna(0.0).to_numpy()

            ax.barh(
                y + offset,
                values,
                height=bar_height,
                xerr=xerr,
                capsize=3 if xerr is not None else 0,
                label=type_labels[prediction_type],
                color=colors[prediction_type],
                alpha=0.9,
            )

        labels = [
            (
                f"{species} (n={int(species_counts.loc[species])}, "
                f"n_unique_taxid={n_unique_taxids_by_species_group.get(species, 0)})"
            )
            for species in species_order
        ]
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.invert_yaxis()
        ax.set_title(f"{endpoint}: median AE by species group")
        ax.grid(axis="x", alpha=0.25)

    axes[-1].set_xlabel("Fold-mean median absolute error in log10 concentration")
    handles, labels = [], []
    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            break
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(len(handles), 3), frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()

    return fig, axes, summary


# This replaces only the projected pretrained-taxid branch of the metadata encoder.
avgtaxid_results_df = run_average_taxid_ablation()
avgtaxid_fold_summary, avgtaxid_endpoint_species_summary = summarize_avgtaxid_endpoint_species(avgtaxid_results_df)
fig, axes, avgtaxid_plot_summary = plot_avgtaxid_endpoint_species_median_ae(
    avgtaxid_results_df,
    endpoints=["EC50"],
    top_species=10,
    min_total_n=20,
    errorbar="std",
)

avgtaxid_sort_cols = [
    "endpoint",
    "species_group_avgtaxid_minus_normal_median_ae",
    "avgtaxid_minus_normal_median_ae",
    "total_n",
]
avgtaxid_sort_cols = [
    col for col in avgtaxid_sort_cols if col in avgtaxid_endpoint_species_summary.columns
]
avgtaxid_endpoint_species_summary.sort_values(
    avgtaxid_sort_cols,
    ascending=[True] + [False] * (len(avgtaxid_sort_cols) - 1),
).head(30)

Unique taxids for average pretrained taxid embedding: 8,119
Unique taxids by species group: algae=442, amphibians=170, birds=115, cnidarians and bryozoans=68, crustaceans=608, cyanobacteria=91, echinoderms=48, fish=651, fungi=571, insects=2,231, mollusks=402, other mammals=46, plants=1,922, protozoans=113, reptiles=30, rodents=65, rotifers=72, tunicates and sponges=15, unmapped=53, worms=406
Loaded model from /cephyr/users/vollmers/Alvis/vollmers/gnn-thesis/gnn-thesis/outputs/models/afp-11M-10fold/AttentiveFP-11M-fold0.pt
Fold 0: normal median AE=0.4603, avg-taxid median AE=0.9377, species-group avg-taxid median AE=0.5864
Loaded model from /cephyr/users/vollmers/Alvis/vollmers/gnn-thesis/gnn-thesis/outputs/models/afp-11M-10fold/AttentiveFP-11M-fold1.pt
Fold 1: normal median AE=0.5035, avg-taxid median AE=0.9827, species-group avg-taxid median AE=0.6089
Loaded model from /cephyr/users/vollmers/Alvis/vollmers/gnn-thesis/gnn-thesis/outputs/models/afp-11M-10fold/AttentiveFP-11M-fold2.pt
Fo

,endpoint,species_group,prediction_type,median_ae_mean,median_ae_std,median_ae_sem,median_ae_min,median_ae_max,n_folds,total_n,avgtaxid_minus_normal_median_ae,species_group_avgtaxid_minus_normal_median_ae
51,EC10,unmapped,average_taxid,1.509179,0.411613,0.130163,0.700745,2.004959,10,392,1.071158,0.579715
52,EC10,unmapped,normal,0.438021,0.094599,0.029915,0.341324,0.676897,10,392,1.071158,0.579715
53,EC10,unmapped,species_group_average_taxid,1.017736,0.122295,0.038673,0.797375,1.241872,10,392,1.071158,0.579715
39,EC10,protozoans,average_taxid,1.563686,0.359717,0.113753,0.891692,2.215668,10,179,0.878965,0.420864
40,EC10,protozoans,normal,0.684721,0.163410,0.051675,0.489294,1.061298,10,179,0.878965,0.420864
41,EC10,protozoans,species_group_average_taxid,1.105584,0.480372,0.151907,0.570050,2.007623,10,179,0.878965,0.420864
27,EC10,insects,average_taxid,1.198446,0.304292,0.096225,0.765919,1.743067,10,1088,0.561555,0.365606
28,EC10,insects,normal,0.636891,0.213647,0.067561,0.377895,1.148102,10,1088,0.561555,0.365606
29,EC10,insects,species_group_average_taxid,1.002497,0.319317,0.100977,0.646518,1.490135,10,1088,0.561555,0.365606
45,EC10,rotifers,average_taxid,1.461360,1.038731,0.328476,0.213192,3.060975,10,195,0.826595,0.363330


# BRICS fragment toxicity scan

In [14]:
from IPython.display import display
from rdkit import Chem

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.visualization.interpretability import get_brics_atom_groups, group_shapley_permutation

# Change these three filters for the slice you want to analyze.
# Values can be strings or lists, for example:
# {"conc_unit": "mg/L", "species_group": ["fish", "crustaceans", "algae"], "endpoint": "EC50"}
# {"conc_unit": "mg/kg", "species_group": "rodents", "endpoint": "EC50"}
FRAGMENT_FILTERS = {
    "conc_unit": "mg/kg",
    "species_group": "rodents",
    "endpoint": "EC50",
}

# Start small. Set FRAGMENT_ANALYSIS_FOLDS=None to sample across all validation folds.
FRAGMENT_ANALYSIS_FOLDS = [fold_id]
N_PER_TOXICITY_BAND = 100
MAX_ABS_ERROR_LOG10C = 0.30

# Reliability/speed guards. Raise these once the first run looks sensible.
MIN_BRICS_GROUPS = 2
MAX_BRICS_GROUPS = 12
MAX_MOLECULE_ATOMS = 80
MIN_FRAGMENT_OCCURRENCES = 2
GROUP_SHAPLEY_N_SAMPLES = 64
DEDUPLICATE_BY_SMILES = True
FRAGMENT_RANDOM_STATE = random_state

FRAGMENT_MODEL_DIR = PROJECT_ROOT / "outputs" / "models" / "afp-11M-10fold"
FRAGMENT_CHECKPOINT_TEMPLATE = "AttentiveFP-11M-fold{fold}.pt"


In [15]:
def _filter_values(values):
    if values is None:
        return []
    if isinstance(values, (list, tuple, set, pd.Index, np.ndarray)):
        return [value for value in values if value is not None]
    return [values]


def _string_key_series(series):
    return series.astype("string").str.strip().str.lower()


def filter_fragment_prediction_rows(pred_df, filters, folds=None):
    work = pred_df.copy()
    work["row_id"] = pd.to_numeric(work["row_id"], errors="coerce").astype("Int64")
    work["fold"] = pd.to_numeric(work["fold"], errors="coerce").astype("Int64")
    work = work.dropna(subset=["row_id", "fold", "pred_log10c"]).copy()
    work["row_id"] = work["row_id"].astype(int)
    work["fold"] = work["fold"].astype(int)

    for col, raw_values in filters.items():
        values = _filter_values(raw_values)
        if not values:
            continue
        if col not in work.columns:
            raise KeyError(f"{col!r} is missing from the prediction dataframe.")
        wanted = {str(value).strip().lower() for value in values}
        work = work[_string_key_series(work[col]).isin(wanted)].copy()

    if folds is not None:
        folds = [int(fold) for fold in _filter_values(folds)]
        work = work[work["fold"].isin(folds)].copy()

    target_col = "actual_log10c" if "actual_log10c" in work.columns else "log10c"
    if target_col not in work.columns:
        raise KeyError("Expected either 'actual_log10c' or 'log10c' in the prediction dataframe.")

    work["actual_log10c"] = pd.to_numeric(work[target_col], errors="coerce")
    work["pred_log10c"] = pd.to_numeric(work["pred_log10c"], errors="coerce")
    if "abs_error_log10c" in work.columns:
        work["abs_error_log10c"] = pd.to_numeric(work["abs_error_log10c"], errors="coerce")
    else:
        work["abs_error_log10c"] = np.nan

    missing_error = work["abs_error_log10c"].isna()
    work.loc[missing_error, "abs_error_log10c"] = (
        work.loc[missing_error, "pred_log10c"] - work.loc[missing_error, "actual_log10c"]
    ).abs()

    return work.dropna(subset=["actual_log10c", "pred_log10c", "abs_error_log10c"]).copy()


_brics_diagnostic_cache = {}


def cached_brics_diagnostics(smiles):
    if smiles in _brics_diagnostic_cache:
        return _brics_diagnostic_cache[smiles]

    mol = Chem.MolFromSmiles(str(smiles)) if isinstance(smiles, str) else None
    if mol is None:
        result = {
            "n_atoms": np.nan,
            "n_brics_groups": np.nan,
            "brics_group_atom_sizes": tuple(),
        }
    else:
        groups = get_brics_atom_groups(mol)
        result = {
            "n_atoms": int(mol.GetNumAtoms()),
            "n_brics_groups": int(len(groups)),
            "brics_group_atom_sizes": tuple(len(group) for group in groups),
        }

    _brics_diagnostic_cache[smiles] = result
    return result


def add_brics_diagnostics(frame):
    diagnostics = pd.DataFrame(
        [cached_brics_diagnostics(smiles) for smiles in frame["SMILES"]],
        index=frame.index,
    )
    return pd.concat([frame, diagnostics], axis=1)


def prepare_fragment_candidates(
    pred_df,
    filters,
    folds=None,
    max_abs_error=0.30,
    min_brics_groups=2,
    max_brics_groups=12,
    max_atoms=80,
    deduplicate_by_smiles=True,
):
    filtered = filter_fragment_prediction_rows(pred_df, filters, folds=folds)
    accurate = filtered[filtered["abs_error_log10c"] <= max_abs_error].copy()

    if deduplicate_by_smiles:
        accurate = (
            accurate.sort_values(["SMILES", "abs_error_log10c", "pred_log10c"])
            .drop_duplicates("SMILES", keep="first")
            .copy()
        )

    if accurate.empty:
        raise ValueError(
            "No rows survived the filter and accuracy threshold. Try a larger "
            "MAX_ABS_ERROR_LOG10C, a broader species_group list, or FRAGMENT_ANALYSIS_FOLDS=None."
        )

    with_diagnostics = add_brics_diagnostics(accurate)
    candidates = with_diagnostics[
        with_diagnostics["n_atoms"].le(max_atoms)
        & with_diagnostics["n_brics_groups"].between(min_brics_groups, max_brics_groups)
    ].copy()

    if candidates.empty:
        raise ValueError(
            "Rows matched the filter, but none survived the BRICS/atom-count guards. "
            "Try increasing MAX_BRICS_GROUPS/MAX_MOLECULE_ATOMS or lowering MIN_BRICS_GROUPS."
        )

    print(f"Filtered rows: {len(filtered):,}")
    print(f"Rows with abs error <= {max_abs_error:.2f}: {len(accurate):,}")
    print(f"Rows after BRICS guards: {len(candidates):,}")
    print(
        "Prediction range in candidates: "
        f"{candidates['pred_log10c'].min():.3f} to {candidates['pred_log10c'].max():.3f} log10 concentration"
    )
    return candidates


def select_toxicity_band_rows(candidates, n_per_band=10):
    candidates = candidates.copy()
    median_pred = candidates["pred_log10c"].median()
    selected = []
    used_row_ids = set()

    most = candidates.sort_values(["pred_log10c", "abs_error_log10c"], ascending=[True, True]).head(n_per_band).copy()
    most["toxicity_band"] = "most_toxic"
    selected.append(most)
    used_row_ids.update(most["row_id"].tolist())

    remaining = candidates[~candidates["row_id"].isin(used_row_ids)].copy()
    least = remaining.sort_values(["pred_log10c", "abs_error_log10c"], ascending=[False, True]).head(n_per_band).copy()
    least["toxicity_band"] = "least_toxic"
    selected.append(least)
    used_row_ids.update(least["row_id"].tolist())

    remaining = candidates[~candidates["row_id"].isin(used_row_ids)].copy()
    mid = remaining.assign(_mid_distance=(remaining["pred_log10c"] - median_pred).abs())
    mid = mid.sort_values(["_mid_distance", "abs_error_log10c"], ascending=[True, True]).head(n_per_band).copy()
    mid = mid.drop(columns=["_mid_distance"])
    mid["toxicity_band"] = "mid_toxic"
    selected.append(mid)

    out = pd.concat(selected, ignore_index=True)
    band_order = pd.CategoricalDtype(["most_toxic", "mid_toxic", "least_toxic"], ordered=True)
    out["toxicity_band"] = out["toxicity_band"].astype(band_order)
    return out.sort_values(["toxicity_band", "pred_log10c", "abs_error_log10c"]).reset_index(drop=True)


def load_fragment_analysis_model(fold):
    model_for_fold, *_ = build_model()
    checkpoint_path = FRAGMENT_MODEL_DIR / FRAGMENT_CHECKPOINT_TEMPLATE.format(fold=int(fold))
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model_for_fold.load_state_dict(checkpoint["model_state_dict"])
    model_for_fold = model_for_fold.to(device)
    model_for_fold.eval()
    return model_for_fold, checkpoint_path


def brics_fragment_smiles(mol, atom_indices):
    atom_indices = sorted(int(atom_idx) for atom_idx in atom_indices)
    return Chem.MolFragmentToSmiles(
        mol,
        atomsToUse=atom_indices,
        canonical=True,
        isomericSmiles=True,
    )


def explain_selected_brics_groups(selected_rows, n_samples=64):
    if selected_rows.empty:
        return pd.DataFrame()

    torch.manual_seed(int(FRAGMENT_RANDOM_STATE))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(FRAGMENT_RANDOM_STATE))

    result_rows = []
    selected_rows = selected_rows.sort_values(["fold", "toxicity_band", "pred_log10c"]).reset_index(drop=True)

    for fold, fold_rows in selected_rows.groupby("fold", sort=True):
        model_for_fold, checkpoint_path = load_fragment_analysis_model(fold)
        print(f"Loaded fold {fold} model for {len(fold_rows)} selected rows: {checkpoint_path.name}")

        for local_idx, row in fold_rows.reset_index(drop=True).iterrows():
            row_id = int(row["row_id"])
            graph = graphs[row_id].clone().to(device)
            mol = Chem.MolFromSmiles(row["SMILES"])
            groups = get_brics_atom_groups(mol)

            shapley_log10c = group_shapley_permutation(
                model_for_fold,
                graph,
                groups,
                n_samples=n_samples,
            ).detach().cpu().numpy()
            tox_shapley_log10c = -shapley_log10c
            top_toxic_group_idx = int(np.argmax(tox_shapley_log10c))

            for group_idx, (atom_indices, shap_value, tox_value) in enumerate(
                zip(groups, shapley_log10c, tox_shapley_log10c)
            ):
                n_group_atoms = len(atom_indices)
                result_rows.append(
                    {
                        "row_id": row_id,
                        "fold": int(fold),
                        "toxicity_band": row["toxicity_band"],
                        "SMILES": row["SMILES"],
                        "chemical_name": row.get("chemical_name", np.nan),
                        "CAS": row.get("CAS", np.nan),
                        "conc_unit": row.get("conc_unit", np.nan),
                        "species_group": row.get("species_group", np.nan),
                        "endpoint": row.get("endpoint", np.nan),
                        "effect": row.get("effect", np.nan),
                        "actual_log10c": row["actual_log10c"],
                        "pred_log10c": row["pred_log10c"],
                        "abs_error_log10c": row["abs_error_log10c"],
                        "n_brics_groups": len(groups),
                        "n_atoms": row["n_atoms"],
                        "group_idx": group_idx,
                        "fragment_smiles": brics_fragment_smiles(mol, atom_indices),
                        "n_group_atoms": n_group_atoms,
                        "group_shapley_log10c": float(shap_value),
                        "tox_shapley_log10c": float(tox_value),
                        "tox_shapley_per_atom_log10c": float(tox_value) / max(n_group_atoms, 1),
                        "is_top_toxic_group": group_idx == top_toxic_group_idx,
                    }
                )

            print(
                f"  {local_idx + 1:>2}/{len(fold_rows)} row_id={row_id} "
                f"band={row['toxicity_band']} groups={len(groups)} "
                f"pred={row['pred_log10c']:.3f} actual={row['actual_log10c']:.3f}"
            )

        del model_for_fold
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(result_rows)


def summarize_brics_group_toxicity(fragment_df, min_occurrences=2):
    if fragment_df.empty:
        empty = pd.DataFrame()
        return empty, empty, empty, empty

    def sem(values):
        values = pd.Series(values).dropna()
        if len(values) <= 1:
            return 0.0
        return values.std(ddof=1) / np.sqrt(len(values))

    summary = (
        fragment_df.groupby("fragment_smiles", dropna=False)
        .agg(
            n_occurrences=("tox_shapley_log10c", "size"),
            n_molecules=("row_id", "nunique"),
            n_folds=("fold", "nunique"),
            n_bands=("toxicity_band", "nunique"),
            mean_tox_shapley_log10c=("tox_shapley_log10c", "mean"),
            std_tox_shapley_log10c=("tox_shapley_log10c", "std"),
            sem_tox_shapley_log10c=("tox_shapley_log10c", sem),
            median_tox_shapley_log10c=("tox_shapley_log10c", "median"),
            min_tox_shapley_log10c=("tox_shapley_log10c", "min"),
            max_tox_shapley_log10c=("tox_shapley_log10c", "max"),
            mean_tox_shapley_per_atom_log10c=("tox_shapley_per_atom_log10c", "mean"),
            std_tox_shapley_per_atom_log10c=("tox_shapley_per_atom_log10c", "std"),
            mean_group_atoms=("n_group_atoms", "mean"),
            positive_fraction=("tox_shapley_log10c", lambda values: float((values > 0).mean())),
            n_top_toxic_group=("is_top_toxic_group", "sum"),
            mean_pred_log10c=("pred_log10c", "mean"),
            mean_actual_log10c=("actual_log10c", "mean"),
            mean_abs_error_log10c=("abs_error_log10c", "mean"),
        )
        .reset_index()
    )

    fill_zero_cols = [col for col in summary.columns if col.startswith("std_") or col.startswith("sem_")]
    summary[fill_zero_cols] = summary[fill_zero_cols].fillna(0.0)
    summary["top_toxic_group_rate"] = summary["n_top_toxic_group"] / summary["n_occurrences"].clip(lower=1)
    summary["has_mixed_toxicity_sign"] = summary["positive_fraction"].between(0.05, 0.95)

    valid = summary[summary["n_occurrences"] >= min_occurrences].copy()
    toxic_ranked = valid.sort_values(
        ["mean_tox_shapley_per_atom_log10c", "mean_tox_shapley_log10c", "n_occurrences"],
        ascending=[False, False, False],
    )
    raw_toxic_ranked = valid.sort_values(
        ["mean_tox_shapley_log10c", "mean_tox_shapley_per_atom_log10c", "n_occurrences"],
        ascending=[False, False, False],
    )
    context_ranked = valid.sort_values(
        ["std_tox_shapley_log10c", "has_mixed_toxicity_sign", "n_occurrences"],
        ascending=[False, False, False],
    )
    return summary, toxic_ranked, raw_toxic_ranked, context_ranked


def summarize_brics_by_band(fragment_df, min_occurrences=2):
    if fragment_df.empty:
        return pd.DataFrame()

    band_summary = (
        fragment_df.groupby(["toxicity_band", "fragment_smiles"], observed=True, dropna=False)
        .agg(
            n_occurrences=("tox_shapley_log10c", "size"),
            n_molecules=("row_id", "nunique"),
            mean_tox_shapley_log10c=("tox_shapley_log10c", "mean"),
            std_tox_shapley_log10c=("tox_shapley_log10c", "std"),
            mean_tox_shapley_per_atom_log10c=("tox_shapley_per_atom_log10c", "mean"),
            positive_fraction=("tox_shapley_log10c", lambda values: float((values > 0).mean())),
            n_top_toxic_group=("is_top_toxic_group", "sum"),
        )
        .reset_index()
    )
    band_summary["std_tox_shapley_log10c"] = band_summary["std_tox_shapley_log10c"].fillna(0.0)
    return band_summary[band_summary["n_occurrences"] >= min_occurrences].sort_values(
        ["toxicity_band", "mean_tox_shapley_per_atom_log10c", "n_occurrences"],
        ascending=[True, False, False],
    )


def plot_top_brics_groups(summary_df, top_n=15, value_col="mean_tox_shapley_per_atom_log10c"):
    if summary_df.empty:
        print("No repeated BRICS groups to plot. Lower MIN_FRAGMENT_OCCURRENCES or sample more molecules.")
        return None, None

    plot_df = summary_df.head(top_n).iloc[::-1].copy()
    error_col = "std_tox_shapley_per_atom_log10c" if value_col.endswith("per_atom_log10c") else "std_tox_shapley_log10c"

    fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(plot_df))))
    ax.barh(plot_df["fragment_smiles"], plot_df[value_col], xerr=plot_df[error_col], capsize=3)
    ax.axvline(0, color="black", linewidth=1, alpha=0.5)
    ax.set_xlabel("Mean toxic contribution (-Shapley, log10c units)")
    ax.set_ylabel("BRICS fragment")
    ax.set_title("BRICS fragments ranked by model-toxic contribution")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    plt.show()
    return fig, ax


In [16]:
fragment_candidates = prepare_fragment_candidates(
    afp_df,
    FRAGMENT_FILTERS,
    folds=FRAGMENT_ANALYSIS_FOLDS,
    max_abs_error=MAX_ABS_ERROR_LOG10C,
    min_brics_groups=MIN_BRICS_GROUPS,
    max_brics_groups=MAX_BRICS_GROUPS,
    max_atoms=MAX_MOLECULE_ATOMS,
    deduplicate_by_smiles=DEDUPLICATE_BY_SMILES,
)

selected_fragment_rows = select_toxicity_band_rows(
    fragment_candidates,
    n_per_band=N_PER_TOXICITY_BAND,
)

sample_cols = [
    "toxicity_band",
    "row_id",
    "fold",
    "chemical_name",
    "SMILES",
    "conc_unit",
    "species_group",
    "endpoint",
    "actual_log10c",
    "pred_log10c",
    "abs_error_log10c",
    "n_atoms",
    "n_brics_groups",
    "brics_group_atom_sizes",
]
sample_cols = [col for col in sample_cols if col in selected_fragment_rows.columns]
display(selected_fragment_rows[sample_cols])

brics_fragment_contributions = explain_selected_brics_groups(
    selected_fragment_rows,
    n_samples=GROUP_SHAPLEY_N_SAMPLES,
)

(
    brics_group_summary,
    brics_most_toxic_groups_per_atom,
    brics_most_toxic_groups_raw,
    brics_context_dependent_groups,
) = summarize_brics_group_toxicity(
    brics_fragment_contributions,
    min_occurrences=MIN_FRAGMENT_OCCURRENCES,
)

brics_band_summary = summarize_brics_by_band(
    brics_fragment_contributions,
    min_occurrences=MIN_FRAGMENT_OCCURRENCES,
)

print("Most model-toxic repeated BRICS groups, normalized per atom:")
display(brics_most_toxic_groups_per_atom.head(20))

print("Most model-toxic repeated BRICS groups, raw group contribution:")
display(brics_most_toxic_groups_raw.head(20))

print("Most context-dependent repeated BRICS groups by Shapley std:")
display(brics_context_dependent_groups.head(20))

print("Band-specific repeated BRICS group summary:")
display(brics_band_summary.head(40))

plot_top_brics_groups(brics_most_toxic_groups_per_atom, top_n=15)


Filtered rows: 10,504
Rows with abs error <= 0.30: 3,879
Rows after BRICS guards: 3,497
Prediction range in candidates: -1.169 to 3.754 log10 concentration


,toxicity_band,row_id,fold,chemical_name,SMILES,conc_unit,species_group,endpoint,actual_log10c,pred_log10c,abs_error_log10c,n_atoms,n_brics_groups,brics_group_atom_sizes
0,most_toxic,7702,0,"Ammonium, dimethyl(2-((ethoxymethylphosphinyl)thio)ethyl)phenyl-, methyl sul...",CCOP(C)(=O)SCC[N+](C)(C)c1ccccc1.COS(=O)(=O)[O-],mg/kg,rodents,EC50,-1.455932,-1.168817,0.287115,24,6,"(2, 5, 2, 3, 6, 6)"
1,most_toxic,8242,0,"Ammonium, (2-hydroxypropyl)trimethyl-, iodide, methylphosphonofluoridate, (e...",CC(C[N+](C)(C)C)OP(C)(=O)F.[I-],mg/kg,rodents,EC50,-1.154902,-1.155828,0.000926,13,4,"(3, 4, 5, 1)"
2,most_toxic,73200,0,"Phosphonothioic acid, methyl-, S-(2-(dimethylamino)ethyl) O-(2-methylpropyl)...",CC(C)COP(C)(=O)SCCN(C)C,mg/kg,rodents,EC50,-1.102373,-1.128312,0.025939,14,4,"(4, 5, 2, 3)"
3,most_toxic,7796,0,"Ammonium, dipropyl((3-hydroxy-2-pyridyl)methyl)methyl-, bromide, dimethylcar...",CCC[N+](C)(CCC)Cc1ncccc1OC(=O)N(C)C.[Br-],mg/kg,rodents,EC50,-0.795880,-0.594691,0.201189,22,9,"(3, 2, 3, 1, 6, 1, 2, 3, 1)"
4,most_toxic,83840,0,"1,3-Propanediol, 2-(tert-butyl)-2-(hydroxymethyl)-, cyclic phosphite (1:1)",CC(C)(C)C12COP(OC1)OC2,mg/kg,rodents,EC50,-0.677781,-0.568506,0.109275,12,2,"(4, 8)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,least_toxic,35901,0,"Cyclohexanone, 2-(1-methylpropyl)-",CCC(C)C1CCCCC1=O,mg/kg,rodents,EC50,3.380211,3.583918,0.203707,11,2,"(4, 7)"
296,least_toxic,2960,0,"Acetic acid, (1,1-dimethyl-3-phenylpropyl) ester",CC(=O)OC(C)(C)CCc1ccccc1,mg/kg,rodents,EC50,3.685742,3.609425,0.076317,15,4,"(3, 1, 5, 6)"
297,least_toxic,103458,0,"Succinic acid, dipropyl ester",CCCOC(=O)CCC(=O)OCCC,mg/kg,rodents,EC50,3.812245,3.670938,0.141306,14,5,"(3, 1, 6, 1, 3)"
298,least_toxic,19461,0,"Benzoic acid, 3,3'-(adipoyldiimino)bis(2,4,6-triiodo-, disodium salt",O=C(CCCCC(=O)Nc1c(I)cc(I)c(C(=O)[O-])c1I)Nc1c(I)cc(I)c(C(=O)[O-])c1I.[Na+].[...,mg/kg,rodents,EC50,3.531479,3.683555,0.152076,36,9,"(8, 1, 9, 3, 1, 9, 3, 1, 1)"


Loaded fold 0 model for 300 selected rows: AttentiveFP-11M-fold0.pt
   1/300 row_id=7702 band=most_toxic groups=6 pred=-1.169 actual=-1.456
   2/300 row_id=8242 band=most_toxic groups=4 pred=-1.156 actual=-1.155
   3/300 row_id=73200 band=most_toxic groups=4 pred=-1.128 actual=-1.102
   4/300 row_id=7796 band=most_toxic groups=9 pred=-0.595 actual=-0.796
   5/300 row_id=83840 band=most_toxic groups=2 pred=-0.569 actual=-0.678
   6/300 row_id=8045 band=most_toxic groups=7 pred=-0.491 actual=-0.398
   7/300 row_id=5139 band=most_toxic groups=12 pred=-0.374 actual=-0.658
   8/300 row_id=68743 band=most_toxic groups=5 pred=-0.262 actual=-0.354
   9/300 row_id=37359 band=most_toxic groups=6 pred=-0.212 actual=0.086
  10/300 row_id=111336 band=most_toxic groups=2 pred=-0.139 actual=-0.125
  11/300 row_id=75145 band=most_toxic groups=7 pred=-0.127 actual=-0.222
  12/300 row_id=90198 band=most_toxic groups=5 pred=-0.065 actual=0.000
  13/300 row_id=108261 band=most_toxic groups=6 pred=-0.064 a

,fragment_smiles,n_occurrences,n_molecules,n_folds,n_bands,mean_tox_shapley_log10c,std_tox_shapley_log10c,sem_tox_shapley_log10c,median_tox_shapley_log10c,min_tox_shapley_log10c,max_tox_shapley_log10c,mean_tox_shapley_per_atom_log10c,std_tox_shapley_per_atom_log10c,mean_group_atoms,positive_fraction,n_top_toxic_group,mean_pred_log10c,mean_actual_log10c,mean_abs_error_log10c,top_toxic_group_rate,has_mixed_toxicity_sign
150,CP(=O)(O)S,3,3,1,1,2.647982,0.382464,0.220816,2.671064,2.254500,3.018382,0.529596,0.076493,5.0,1.000000,3,-0.696241,-0.736019,0.151632,1.000000,False
352,c1cnoc1,2,2,1,1,1.597179,0.360456,0.254881,1.597179,1.342298,1.852060,0.319436,0.072091,5.0,1.000000,2,0.035710,0.000000,0.035710,1.000000,False
221,Nc1cn[nH]c1,3,3,1,1,1.828325,0.062037,0.035817,1.827078,1.766921,1.890976,0.304721,0.010340,6.0,1.000000,3,0.004166,0.000000,0.047203,1.000000,False
167,Cc1cc(N)c[nH]1,2,2,1,1,1.972771,0.020055,0.014181,1.972771,1.958589,1.986952,0.281824,0.002865,7.0,1.000000,2,0.062173,0.000000,0.062173,1.000000,False
207,N,92,74,1,3,0.279080,0.279035,0.029091,0.195652,-0.156510,1.261503,0.279080,0.279035,1.0,0.902174,36,1.892971,1.904436,0.135156,0.391304,True
44,CC(=NO)c1cccs1,3,3,1,1,2.422881,0.191017,0.110284,2.363386,2.268691,2.636566,0.269209,0.021224,9.0,1.000000,3,0.070337,0.000000,0.070337,1.000000,False
255,O=CC(C=O)=C1SCCS1,2,2,1,1,2.634943,0.108562,0.076765,2.634943,2.558178,2.711708,0.263494,0.010856,10.0,1.000000,2,0.021345,0.000000,0.021345,1.000000,False
204,Fc1cccnc1,2,2,1,1,1.655384,0.248031,0.175385,1.655384,1.479999,1.830768,0.236483,0.035433,7.0,1.000000,2,-0.002211,0.000000,0.019069,1.000000,False
247,O=C1CSC(=S)N1,4,4,1,1,1.636146,0.512005,0.256003,1.788374,0.938749,2.029089,0.233735,0.073144,7.0,1.000000,3,0.018453,0.000000,0.046431,0.750000,False
309,S,12,12,1,3,0.233371,0.388704,0.112209,0.104655,-0.162442,0.941211,0.233371,0.388704,1.0,0.666667,2,1.962504,1.972444,0.078745,0.166667,True


Most model-toxic repeated BRICS groups, raw group contribution:


...

Most context-dependent repeated BRICS groups by Shapley std:


,fragment_smiles,n_occurrences,n_molecules,n_folds,n_bands,mean_tox_shapley_log10c,std_tox_shapley_log10c,sem_tox_shapley_log10c,median_tox_shapley_log10c,min_tox_shapley_log10c,max_tox_shapley_log10c,mean_tox_shapley_per_atom_log10c,std_tox_shapley_per_atom_log10c,mean_group_atoms,positive_fraction,n_top_toxic_group,mean_pred_log10c,mean_actual_log10c,mean_abs_error_log10c,top_toxic_group_rate,has_mixed_toxicity_sign
276,O=P(O)(O)S,3,3,1,2,0.993014,1.094074,0.631664,1.500557,-0.262654,1.741139,0.198603,0.218815,5.0,0.666667,2,1.038069,1.076646,0.110374,0.666667,True
213,NC(N)=S,2,2,1,2,0.678845,0.751554,0.531429,0.678845,0.147416,1.210275,0.169711,0.187889,4.0,1.000000,1,1.108852,1.127636,0.045608,0.500000,False
331,c1ccc2[nH]ccc2c1,2,2,1,1,0.712540,0.698052,0.493597,0.712540,0.218943,1.206137,0.079171,0.077561,9.0,1.000000,1,0.119580,0.039591,0.079989,0.500000,False
19,C1CNCCN1,6,6,1,2,0.693929,0.678931,0.277172,0.612962,-0.048971,1.655745,0.115655,0.113155,6.0,0.833333,5,1.185251,1.192543,0.124411,0.833333,True
241,O=C1CCCO1,3,3,1,2,-0.322192,0.615647,0.355444,-0.615992,-0.735901,0.385316,-0.053699,0.102608,6.0,0.333333,0,2.112498,2.126737,0.018455,0.000000,True
20,C1COCCN1,7,7,1,2,0.273922,0.593662,0.224383,0.023342,-0.280311,1.438451,0.045654,0.098944,6.0,0.571429,2,1.331276,1.289563,0.074751,0.285714,True
23,C1COCOC1,2,2,1,2,0.087998,0.567018,0.400942,0.087998,-0.312944,0.488941,0.014666,0.094503,6.0,0.500000,0,1.368732,1.435900,0.067168,0.000000,True
247,O=C1CSC(=S)N1,4,4,1,1,1.636146,0.512005,0.256003,1.788374,0.938749,2.029089,0.233735,0.073144,7.0,1.000000,3,0.018453,0.000000,0.046431,0.750000,False
200,Cn1ncccc1=O,2,2,1,1,0.654943,0.491540,0.347571,0.654943,0.307371,1.002514,0.081868,0.061443,8.0,1.000000,0,0.507959,0.715682,0.207723,0.000000,False
169,Cc1ccc(O)cc1,3,3,1,1,0.915788,0.469182,0.270883,0.824007,0.499278,1.424079,0.114474,0.058648,8.0,1.000000,1,0.044595,0.000000,0.044595,0.333333,False


Band-specific repeated BRICS group summary:


,toxicity_band,fragment_smiles,n_occurrences,n_molecules,mean_tox_shapley_log10c,std_tox_shapley_log10c,mean_tox_shapley_per_atom_log10c,positive_fraction,n_top_toxic_group
68,least_toxic,N,33,28,0.113454,0.101106,0.113454,0.878788,21
60,least_toxic,Cl,2,2,0.071214,0.051165,0.071214,1.000000,2
87,least_toxic,O=CCCl,2,2,0.253303,0.335696,0.063326,1.000000,1
1,least_toxic,C,5,5,0.048509,0.047544,0.048509,0.800000,3
89,least_toxic,O=CCl,2,2,0.129078,0.007978,0.043026,1.000000,2
91,least_toxic,O=CS,2,1,0.121202,0.105675,0.040401,1.000000,0
52,least_toxic,Cc1cccc(=O)[nH]1,2,2,0.243115,0.189563,0.030389,1.000000,2
76,least_toxic,O,62,36,0.016856,0.096855,0.016856,0.532258,22
45,least_toxic,CN,3,3,0.028507,0.082793,0.014254,0.666667,2
121,least_toxic,[O-2],5,1,0.007410,0.019465,0.007410,0.400000,1


(<Figure size 1000x600 with 1 Axes>,
 <Axes: title={'center': 'BRICS fragments ranked by model-toxic contribution'}, xlabel='Mean toxic contribution (-Shapley, log10c units)', ylabel='BRICS fragment'>)